In [ ]:
using LowLevelFEM, LinearAlgebra

In [ ]:
openGeometry("boxes.geo")

In [ ]:
#openPreProcessor()

In [ ]:
mat = Material("body")
U = Field([mat], type=:VectorField, dim=3, fieldName=:u);

In [ ]:
bc_bottom = BoundaryCondition("bottom", ux=0, uy=0, uz=0)
bc_top = BoundaryCondition("top", ux=0, uz=0, uy=(x,y,z)->-x*(x-10) * z*(z-10) / 4250)

K = ∫(SymGrad(U) ⋅ D(:Solid, mat) ⋅ SymGrad(U))
f = ∫(U ⋅ [0, 0, 0])

@time u = solveField(Symmetric(K), f, support=[bc_bottom, bc_top])

showDoFResults(u, name="u", factor=1, visible=false)

In [ ]:
contact_pair = contact(u, master="master", slave="slave", cn=1e8)

In [ ]:
support = [bc_bottom, bc_top]
free = freeDoFs(U, support)

p = nothing
rc = nothing
u_it = copy(u)

old_tags = copy(contact_pair.master_element_tags)
old_G = copy(contact_pair.G)

for iter in 1:40

    updateContact!(contact_pair, u_it)

    (; G, C, g) = contact_pair

    nchanged = count(old_tags .!= contact_pair.master_element_tags)

    dG = norm(G - old_G) /
         max(norm(old_G), eps())

    println(
        "master changes = ", nchanged,
        ", dG = ", dG
    )

    old_tags = copy(contact_pair.master_element_tags)
    old_G = copy(G)

    # Penalty contact
    p  = -C * g
    rc = -G' * p
    Kc =  G' * C * G

    # Equilibrium residual and tangent
    r = K * u_it - f + rc
    A = K + Kc

    # Homogeneous Newton correction on prescribed DoFs
    Δu = vectorField(U, "body", [0, 0, 0])
    DoFs(Δu)[free] = -A[free, free] \ DoFs(r)[free]

    r0 = norm(DoFs(r)[free])

    α = 1.0
    u_trial = copy(u_it)
    r_trial = nothing

    while α > 1e-6

        u_trial = u_it + α * Δu

        updateContact!(contact_pair, u_trial)

        (; G, C, g) = contact_pair

        p_trial = -C * g
        rc_trial = -G' * p_trial

        r_trial = K * u_trial - f + rc_trial

        if norm(DoFs(r_trial)[free]) < r0
            break
        end

        α *= 0.5
    end

    u_it = copy(u_trial)
    r = r_trial

    err = α * norm(DoFs(Δu)[free]) /
          max(norm(DoFs(u_it)), eps())

    println(
        "iter = ", iter,
        ", α = ", α,
        ", active = ", count(contact_pair.active),
        ", min gap = ", minimum(contact_pair.gap_values),
        ", error = ", err,
        ", |r| = ", norm(DoFs(r)[free])
    )

    err < 1e-8 && break
end

u = u_it

In [ ]:
showDoFResults(u, name="u cont.", visible=true, factor=1)

In [ ]:
updateContact!(contact_pair, u)
(; G, C, g) = contact_pair

p = -C * g
rc = -G' * p
showElementResults(nodesToElements(rc, onPhysicalGroup="slave"), name="p")

In [ ]:
showElementResults(contact_pair.gap, name="gap")

In [ ]:
openPostProcessor()

Két vagy több párnál majd:

```Julia
contacts = ContactSet(c1, c2, c3)

updateContact!(contacts, u_it)

Kc = sum(c.G' * c.C * c.G for c in contacts)
rc = sum(c.G' * c.C * c.g for c in contacts)

r = K * u_it - f + rc
A = K + Kc
```